In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.utils import resample
from sklearn.tree import DecisionTreeClassifier
import tensorflow as tf
from tensorflow.keras import layers, Model, models
from tensorflow.keras.metrics import BinaryAccuracy
from tensorflow.keras.optimizers import Adam
from tensorflow_addons.metrics import CohenKappa
from tensorflow.keras.constraints import max_norm

In [3]:
def load_data(mode='train'):
    path = "data/20191203/david_20191203_session"
    f_ext = ".npy"
    if mode=='train':
        mode_path = "1"
        load_path = path + mode_path + f_ext
        data = pd.DataFrame(np.load(load_path))
        return data
    elif mode=='test':
        mode_path = "2"
        load_path = path + mode_path + f_ext
        data = pd.DataFrame(np.load(load_path))
        return data
def band_pass_filter(eeg, freq_range):
    info = mne.create_info(64, 512, ch_types=["eeg"] * 64)
    raw = mne.io.RawArray(eeg.T, info)
    raw.filter(freq_range[0], freq_range[1], fir_design='firwin')

    return raw._data.T

In [4]:
def drop_classes(df):
    label_not_inc = [5] #list(range(2,9))
    indexes_to_drop = []
    i = 0
    while i < len(df):
        if df[66].values[i] in label_not_inc:
            list2 = list(range(i, 767+i))
            i += 767
            indexes_to_drop.extend(list2)
        else:
            i += 1
    indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
    df_sliced = df.take(list(indexes_to_keep))

    df_sliced = df_sliced.reset_index(drop=True)
    return df_sliced

In [5]:
def data_win(sfreq, data, asynch_label):
    sampling_window = 2 * sfreq
    shift_length = 1 * sfreq
    t_start = 0

    new_data = []
    labels = []

    while t_start + sampling_window < data.shape[0]:
        new_data.append(data[t_start:t_start+sampling_window, :].T)
        labels.append(asynch_label[t_start:t_start+sampling_window])

        t_start = t_start + shift_length

    return np.array(new_data), np.array(labels)

def transform_label(label_new):
    label = []
    for i in label_new:
        count1 = np.count_nonzero(i==1)
        count9 = np.count_nonzero(i==2)
        if count1 >= 512:
            to_add = 1
        elif count9 >= 512:
            to_add = 2
        else:
            to_add = 0
        label.append(to_add)
        
    label = np.array(label)
    return label

In [6]:
def prune_records(data_new, label):
    train_data = []
    label_data = []

    for i in range(len(label)):
        if label[i] != 0:
            label_data.append(label[i])
            train_data.append(data_new[i])
    label_data = np.array(label_data)
    train_data = np.array(train_data)

    return train_data, label_data

In [7]:
def get_train_data():
    sfreq = 512
    trigger_points = {1:4, 2:4}
    freq_range = [8, 32]
    
    df = load_data(mode='train')
    train_df = drop_classes(df)
    
    train_df = train_df.drop([64, 65], axis=1)
    out_data = band_pass_filter(train_df.iloc[:, :-1].values, freq_range=freq_range)

    X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=train_df.iloc[:, -1].values)
    y = transform_label(y)
    X, y = prune_records(X, y)
    dim1, dim2, dim3 = X.shape
    X_new = X.reshape((dim1, 1, dim2, dim3))
    y = y-1
    
    return X_new, y

def get_eval_data():
    sfreq = 512
    trigger_points = {1:4, 2:4}
    freq_range = [8, 32]
    
    df = load_data(mode='train')
    train_df = drop_classes(df)
    
    train_df = train_df.drop([64, 65], axis=1)
    out_data = band_pass_filter(train_df.iloc[:, :-1].values, freq_range=freq_range)

    X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=train_df.iloc[:, -1].values)
    y = transform_label(y)
    X, y = prune_records(X, y)
    dim1, dim2, dim3 = X.shape
    X_new = X.reshape((dim1, 1, dim2, dim3))
    y = y-1
    
    return X_new, y

In [8]:
def EEGNET(channels=64, samples=1024):
    input1 = layers.Input(shape=(1, channels, samples))
    b1 = layers.Conv2D(8, (1, 512), padding='same', use_bias=False, data_format='channels_first')(input1)
    b1 = layers.BatchNormalization(axis=1)(b1)
    b1 = layers.DepthwiseConv2D((channels, 1), use_bias=False, depth_multiplier=2, depthwise_constraint=max_norm(1.), data_format='channels_first')(b1)
    b1 = layers.BatchNormalization(axis=1)(b1)
    b1 = layers.Activation('elu')(b1)
    b1 = layers.AveragePooling2D((1, 4), data_format='channels_first')(b1)
    b1 = layers.Dropout(0.5)(b1)

    b2 = layers.SeparableConv2D(16, (1, 16), padding='same', use_bias=False, data_format='channels_first')(b1)
    b2 = layers.BatchNormalization(axis=1)(b2)
    b2 = layers.Activation('elu')(b2)
    b2 = layers.AveragePooling2D((1, 8), data_format='channels_first')(b2)
    b2 = layers.Dropout(0.5)(b2)

    flatten = layers.Flatten()(b2)

    dense = layers.Dense(1, kernel_constraint=max_norm(0.25))(flatten)
    activation = layers.Activation('sigmoid')(dense)

    eegnet = Model(inputs = input1, outputs=activation)

    return eegnet

In [9]:
X_train, y_train = get_train_data()
X_eval, y_eval = get_eval_data()

Creating RawArray with float64 data, n_channels=64, n_times=501448
    Range : 0 ... 501447 =      0.000 ...   979.389 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 32 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 32.00 Hz
- Upper transition bandwidth: 8.00 Hz (-6 dB cutoff frequency: 36.00 Hz)
- Filter length: 845 samples (1.650 sec)

Creating RawArray with float64 data, n_channels=64, n_times=501448
    Range : 0 ... 501447 =      0.000 ...   979.389 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 32 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal 

In [10]:
np.save('X_train_20191203.npy', X_train)
np.save('y_train_20191203.npy', y_train)
np.save('X_eval_20191203.npy', X_eval)
np.save('y_eval_20191203.npy', y_eval)

In [ ]:
sum_train_acc = []
sum_test_acc = []
sum_train_kappa = []
sum_test_kappa = []

times = 10

for i in range(times):
    eegnet = EEGNET(channels=64, samples=1024)
    ba = BinaryAccuracy()
    adam = Adam()
    kappa = CohenKappa(num_classes=2)

    eegnet.compile(optimizer=adam, loss='binary_crossentropy', metrics=[ba, kappa])
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    my_callbacks = [
      EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
    ]

    history = eegnet.fit(X_train, y_train, batch_size=25, epochs=500, validation_split=0.25, verbose=2, callbacks=my_callbacks)

    sum_train_acc.append(np.max(history.history['val_binary_accuracy']))
    sum_train_kappa.append(np.max(history.history['val_cohen_kappa']))

    score = eegnet.evaluate(X_eval, y_eval)
    sum_test_acc.append(score[1])
    sum_test_kappa.append(score[2])
    print(score)

    train_acc = []
    train_kappa = []
    test_acc = []
    test_kappa = []

    train_acc.append([np.mean(sum_train_acc), np.std(sum_train_acc)])
    train_kappa.append([np.mean(sum_train_kappa), np.std(sum_train_kappa)])
    test_acc.append([np.mean(sum_test_acc), np.std(sum_test_acc)])
    test_kappa.append([np.mean(sum_test_kappa), np.std(sum_test_kappa)])

print('Train Accuracy --->', train_acc)
print('Train Kappa --->', train_kappa)
print('Test Accuracy --->', test_acc)
print('Test Kappa --->', test_kappa)